# Chapter 1: Why Coordinate-Free Geometry? Singularities, Unwinding & Global Topology

This notebook numerically demonstrates the three core phenomena of Chapter 1 in *A Lie-Interpretation of Robotics and Control*:
1. **Euler-Angle Gimbal Lock ($\theta = \pm 90^\circ$):** Rank deficiency of $E(\theta, \phi)$, condition number blow-up, and unbounded Euler-rate commands.
2. **Unit Quaternion ($S^3$) Unwinding:** How the $S^3 \to SO(3)$ double cover causes $\mathfrak{q}_-(0) = -\mathfrak{q}_+(0)$ to take a $350^\circ$ unwinding path instead of a $10^\circ$ geodesic.
3. **Singularity-Free $SO(3)$ & $S^2$ Geometric Control:** Smooth convergence using $\Psi_R(R, R_d) = \frac{1}{2}\operatorname{Tr}(I_3 - R_d^\top R)$, $e_R = \frac{1}{2}(R_d^\top R - R^\top R_d)^\vee$, and $\Psi_q(q, q_d) = 1 - q \cdot q_d$ directly through gimbal lock and near antipodal equilibria.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True

def hat(x):
    """Hat map (.)^ : R^3 -> so(3) such that hat(x) @ y = np.cross(x, y)."""
    x1, x2, x3 = x
    return np.array([
        [0.0, -x3,  x2],
        [x3,  0.0, -x1],
        [-x2, x1,  0.0]
    ])

def vee(X):
    """Vee map (.)^v : so(3) -> R^3."""
    return np.array([X[2, 1], X[0, 2], X[1, 0]])

def rodrigues_exp(phi):
    """Exponential map exp(hat(phi)) on SO(3) via Rodrigues' formula."""
    theta = np.linalg.norm(phi)
    if theta < 1e-12:
        return np.eye(3) + hat(phi) + 0.5 * (hat(phi) @ hat(phi))
    u_hat = hat(phi / theta)
    return np.eye(3) + np.sin(theta) * u_hat + (1.0 - np.cos(theta)) * (u_hat @ u_hat)

def psi_R(R, R_d):
    """Configuration error Psi_R(R, R_d) = 0.5 * Tr(I_3 - R_d^T R) on SO(3)."""
    return 0.5 * np.trace(np.eye(3) - R_d.T @ R)

def error_R(R, R_d):
    """Attitude error vector e_R = 0.5 * (R_d^T R - R^T R_d)^v."""
    E = R_d.T @ R - R.T @ R_d
    return 0.5 * vee(E)

def psi_q(q, q_d):
    """Configuration error Psi_q(q, q_d) = 1 - q . q_d on S^2."""
    return 1.0 - float(np.dot(q, q_d))

def error_q(q, q_d):
    """Tangent-space error vector e_q = q_d x q on S^2."""
    return np.cross(q_d, q)

# Sanity check: hat(x) @ y == x x y
x_test = np.array([1.2, -0.4, 0.7])
y_test = np.array([-0.3, 2.1, 0.5])
assert np.allclose(hat(x_test) @ y_test, np.cross(x_test, y_test))
assert np.allclose(vee(hat(x_test)), x_test)
print('Core Lie-group utilities verified.')

## 1. Euler-Angle Gimbal Lock ($\theta = \pm 90^\circ$) Jacobian Singularity

For ZYX Euler angles $\eta = (\psi, \theta, \phi)^\top$, the body angular velocity is $\Omega = E(\theta, \phi)\dot{\eta}$ (Eq. 1.5), with $\det(E) = -\cos\theta$. As $\theta \to \pm 90^\circ$, $E$ loses rank and $\dot{\eta} = E^{-1}(\theta, \phi)\Omega$ diverges.

In [ ]:
def euler_E(theta, phi):
    return np.array([
        [-np.sin(theta), 0.0, 1.0],
        [np.cos(theta) * np.sin(phi), np.cos(phi), 0.0],
        [np.cos(theta) * np.cos(phi), -np.sin(phi), 0.0]
    ])

thetas_deg = np.linspace(-89.9, 89.9, 1000)
phi_fixed = np.deg2rad(45.0)
dets = []
conds = []
for th_deg in thetas_deg:
    E = euler_E(np.deg2rad(th_deg), phi_fixed)
    dets.append(np.linalg.det(E))
    conds.append(np.linalg.cond(E))

# Simulate Euler-rate blow-up for constant benign body angular velocity Omega
Omega_const = np.array([0.0, 0.5, 0.5])  # rad/s, ||Omega|| = 0.707 rad/s
dt = 0.001
eta = np.array([0.0, np.deg2rad(70.0), np.deg2rad(45.0)])
t_hist, th_hist, psi_dot_hist, phi_dot_hist = [], [], [], []

for k in range(1000):
    t = k * dt
    E = euler_E(eta[1], eta[2])
    if abs(np.cos(eta[1])) < 0.008:  # stop just before 90 deg
        break
    eta_dot = np.linalg.solve(E, Omega_const)
    t_hist.append(t)
    th_hist.append(np.rad2deg(eta[1]))
    psi_dot_hist.append(eta_dot[0])
    phi_dot_hist.append(eta_dot[2])
    eta = eta + dt * eta_dot

E_89 = euler_E(np.deg2rad(89.0), phi_fixed)
rate_89 = np.linalg.solve(E_89, Omega_const)
print(f'At pitch = 89 deg: det(E) = {np.linalg.det(E_89):.5f}, cond(E) = {np.linalg.cond(E_89):.2f}')
print(f'Euler rates at 89 deg for ||Omega||=0.707 rad/s: psi_dot={rate_89[0]:.2f} rad/s, phi_dot={rate_89[2]:.2f} rad/s')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(thetas_deg, np.abs(dets), label='|det(E)| = cos(theta)', color='tab:blue')
ax1_r = ax1.twinx()
ax1_r.plot(thetas_deg, np.log10(conds), label='log10 cond(E)', color='tab:red', linestyle='--')
ax1.set_xlabel('Pitch angle theta (deg)')
ax1.set_ylabel('|det(E)|', color='tab:blue')
ax1_r.set_ylabel('log10 cond(E)', color='tab:red')
ax1.set_title('Fig. 1.1a: Jacobian Determinant & Conditioning')

ax2.plot(th_hist, psi_dot_hist, label='psi_dot (yaw rate)')
ax2.plot(th_hist, phi_dot_hist, label='phi_dot (roll rate)', linestyle='--')
ax2.set_xlabel('Pitch angle theta (deg)')
ax2.set_ylabel('Euler rate (rad/s)')
ax2.set_title('Fig. 1.1b: Rate Blow-Up Approaching 90 deg')
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Unit Quaternion ($S^3$) Double Cover & Unwinding Phenomenon

Both $+\mathfrak{q}$ and $-\mathfrak{q}$ represent the exact same physical orientation $R(\mathfrak{q}) \in SO(3)$. Under continuous quaternion PD control $\tau = -k_p q_v - k_d \Omega$, an initial state on the $-\mathfrak{q}_I$ hemisphere ($10^\circ$ physically away from $I_3$, i.e., $\theta_{S^3} = 350^\circ$) unwinds through $350^\circ$ instead of taking the $10^\circ$ geodesic.

In [ ]:
def quat_to_R(q):
    q0, qv = q[0], q[1:]
    qh = hat(qv)
    return np.eye(3) + 2.0 * q0 * qh + 2.0 * (qh @ qh)

def quat_mul(q1, q2):
    w1, v1 = q1[0], q1[1:]
    w2, v2 = q2[0], q2[1:]
    return np.concatenate(([w1 * w2 - np.dot(v1, v2)], w1 * v2 + w2 * v1 + np.cross(v1, v2)))

J = np.diag([0.05, 0.05, 0.1])
J_inv = np.linalg.inv(J)
kp, kd = 4.0, 1.2
dt, T = 0.002, 6.0
N = int(T / dt)

def sim_quat(q_init):
    q = q_init.copy()
    Omega = np.zeros(3)
    ang_s3, energy = [], []
    E_cum = 0.0
    for _ in range(N):
        tau = -kp * q[1:] - kd * Omega
        E_cum += np.dot(tau, tau) * dt
        ang_s3.append(np.rad2deg(2.0 * np.arccos(np.clip(q[0], -1.0, 1.0))))
        energy.append(E_cum)
        # Dynamics + kinematics
        Omega_dot = J_inv @ (tau - np.cross(Omega, J @ Omega))
        q_dot = 0.5 * quat_mul(q, np.concatenate(([0.0], Omega)))
        Omega = Omega + dt * Omega_dot
        q = q + dt * q_dot
        q /= np.linalg.norm(q)
    return np.array(ang_s3), np.array(energy)

def sim_so3(R_init, kR=2.0, kW=1.2):
    R = R_init.copy()
    R_d = np.eye(3)
    Omega = np.zeros(3)
    ang_so3, energy = [], []
    E_cum = 0.0
    for _ in range(N):
        eR = error_R(R, R_d)
        tau = -kR * eR - kW * Omega
        E_cum += np.dot(tau, tau) * dt
        theta = np.arccos(np.clip(0.5 * (np.trace(R) - 1.0), -1.0, 1.0))
        ang_so3.append(np.rad2deg(theta))
        energy.append(E_cum)
        Omega_dot = J_inv @ (tau - np.cross(Omega, J @ Omega))
        R = R @ rodrigues_exp(dt * Omega)
        Omega = Omega + dt * Omega_dot
    return np.array(ang_so3), np.array(energy)

alpha = np.deg2rad(-10.0)
q_plus = np.array([np.cos(alpha / 2.0), 0.0, 0.0, np.sin(alpha / 2.0)])
q_minus = -q_plus  # Identical physical rotation in SO(3)!
assert np.allclose(quat_to_R(q_plus), quat_to_R(q_minus))

t_vec = np.arange(N) * dt
ang_plus, E_plus = sim_quat(q_plus)
ang_minus, E_minus = sim_quat(q_minus)
ang_so3, E_so3 = sim_so3(quat_to_R(q_minus))

print(f'Short path (+q) control energy: {E_plus[-1]:.4f}')
print(f'Unwinding (-q) control energy:  {E_minus[-1]:.4f} (ratio = {E_minus[-1]/E_plus[-1]:.1f}x)')
print(f'SO(3) geometric control energy: {E_so3[-1]:.4f}')
assert E_minus[-1] > 20.0 * E_plus[-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(t_vec, ang_plus, label='+q(0): 10 deg short path')
ax1.plot(t_vec, ang_minus, label='-q(0): 350 deg unwinding', linestyle='--')
ax1.plot(t_vec, ang_so3, label='SO(3) geometric (from R(-q))', linestyle=':')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Error Angle (deg)')
ax1.set_title('Fig. 1.2a: S^3 Unwinding vs. SO(3) Geodesic')
ax1.legend()

ax2.plot(t_vec, E_plus, label='+q(0) energy')
ax2.plot(t_vec, E_minus, label='-q(0) unwinding energy', linestyle='--')
ax2.plot(t_vec, E_so3, label='SO(3) geometric energy', linestyle=':')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Cumulative Control Energy')
ax2.set_title('Fig. 1.2b: Wasted Control Effort from Unwinding')
ax2.legend()
plt.tight_layout()
plt.show()

## 3. Singularity-Free $SO(3)$ Geodesic Error $\Psi_R$ & $S^2$ Error $\Psi_q$

We now initialize the $SO(3)$ geometric controller directly at ZYX gimbal lock ($\theta_{\text{pitch}} = 90^\circ$) and near the antipodal saddle ($\theta = 179^\circ$), verifying smooth convergence and machine-precision manifold invariance ($\|R^\top R - I_3\|_F < 10^{-12}$).

In [ ]:
def run_so3_trajectory(R_init, kR=4.0, kW=1.5, T=5.0, dt=0.002):
    N_steps = int(T / dt)
    R = R_init.copy()
    R_d = np.eye(3)
    Omega = np.zeros(3)
    psi_hist, eR_norm_hist, ortho_err_hist = [], [], []
    for _ in range(N_steps):
        psi_hist.append(psi_R(R, R_d))
        eR = error_R(R, R_d)
        eR_norm_hist.append(np.linalg.norm(eR))
        ortho_err_hist.append(np.linalg.norm(R.T @ R - np.eye(3), 'fro'))
        tau = -kR * eR - kW * Omega
        Omega_dot = J_inv @ (tau - np.cross(Omega, J @ Omega))
        R = R @ rodrigues_exp(dt * Omega)
        Omega = Omega + dt * Omega_dot
    return np.arange(N_steps) * dt, np.array(psi_hist), np.array(eR_norm_hist), np.array(ortho_err_hist)

# Case 1: Directly at ZYX gimbal lock (pitch = 90 deg)
R_gimbal = rodrigues_exp(np.array([0.0, np.pi / 2.0, 0.0]))
t_g, psi_g, eR_g, ortho_g = run_so3_trajectory(R_gimbal)

# Case 2: Near antipodal saddle (179 deg)
R_anti = rodrigues_exp(np.array([np.deg2rad(179.0), 0.0, 0.0]))
t_a, psi_a, eR_a, ortho_a = run_so3_trajectory(R_anti)

print(f'At gimbal lock (t=0): Psi_R = {psi_g[0]:.4f}, ||e_R|| = {eR_g[0]:.4f}')
print(f'Final Psi_R (gimbal): {psi_g[-1]:.2e}, Final Psi_R (179 deg): {psi_a[-1]:.2e}')
print(f'Max orthogonality error ||R^T R - I||_F: {max(ortho_g.max(), ortho_a.max()):.2e}')
assert psi_g[-1] < 1e-6 and psi_a[-1] < 1e-4
assert max(ortho_g.max(), ortho_a.max()) < 1e-12

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(t_g, psi_g, label='Psi_R (init: pitch=90 deg gimbal lock)')
ax1.plot(t_a, psi_a, label='Psi_R (init: 179 deg near antipodal)', linestyle='--')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Configuration Error Psi_R')
ax1.set_title('Fig. 1.3a: Singularity-Free SO(3) Convergence')
ax1.legend()

ax2.semilogy(t_g, np.maximum(ortho_g, 1e-16), label='Gimbal lock run')
ax2.semilogy(t_a, np.maximum(ortho_a, 1e-16), label='179 deg run', linestyle='--')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('||R^T R - I_3||_F')
ax2.set_title('Fig. 1.3b: Lie-Group Manifold Invariance')
ax2.legend()
plt.tight_layout()
plt.show()